## 🎯 Learning Objectives
* Understand the components required for a multi-turn conversational chain.
* Implement a `RunnableWithMessageHistory` to manage chat history.
* Integrate a prompt template, LLM, and message history into a LangChain Expression Language (LCEL) chain.
* Test the conversational chain with multiple turns, demonstrating memory retention.


# Exercise: Build a Multi-Turn Conversational Chain

## Task
Your task is to build a multi-turn conversational chain using LangChain Expression Language (LCEL) and `RunnableWithMessageHistory`. This chain should be able to remember previous interactions within a given session, allowing for contextually relevant responses.

## Requirements
1.  **Chat Model:** Use a chat model. For this exercise, we'll provide a mock `ChatOpenAI` instance to avoid API key dependencies, but your solution should be compatible with a real `ChatOpenAI` instance.
2.  **Prompt Template:** Define a `ChatPromptTemplate` that explicitly incorporates `chat_history` (as a `MessagesPlaceholder`) and the current `input` from the user.
3.  **Session History Management:** Implement a `get_session_history` function that returns an instance of `BaseChatMessageHistory` (specifically, `ChatMessageHistory` for in-memory storage) for a given `session_id`. This function will be used by `RunnableWithMessageHistory`.
4.  **LCEL Chain:** Construct a simple LCEL chain that pipes the `ChatPromptTemplate` to your chat model.
5.  **Conversational Wrapper:** Wrap your LCEL chain with `RunnableWithMessageHistory` to enable stateful conversation management.
6.  **Testing:** Demonstrate the functionality by invoking the conversational chain with at least three turns for a single session, showing that the model remembers previous interactions.

## Evaluation Criteria
*   **Correct Prompt Structure:** The `ChatPromptTemplate` correctly uses `MessagesPlaceholder` for `chat_history`.
*   **Functional History Management:** The `get_session_history` function correctly initializes and retrieves `ChatMessageHistory` instances.
*   **Proper `RunnableWithMessageHistory` Usage:** The conversational chain is correctly wrapped and configured with `RunnableWithMessageHistory`.
*   **Contextual Responses:** The chain demonstrates memory by providing contextually relevant responses across multiple turns.
*   **Code Quality:** The code is clean, well-commented, and runnable without errors.


In [ ]:
import os
from typing import Dict, List

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# --- Setup Code: Mock LLM and History Store ---

# In a real scenario, you would initialize ChatOpenAI like this:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# For this exercise, we'll use a mock LLM to avoid API key requirements.
class MockChatOpenAI:
    """A simple mock LLM to simulate chat model behavior."""
    def invoke(self, messages: List[BaseMessage]) -> AIMessage:
        # Extract the last human message to simulate a response
        last_human_message_content = ""
        for msg in reversed(messages):
            if isinstance(msg, HumanMessage):
                last_human_message_content = msg.content
                break
        
        # Simple rule-based responses for demonstration
        if "hello" in last_human_message_content.lower():
            response_content = "Hello there! How can I assist you today?"
        elif "name" in last_human_message_content.lower():
            response_content = "I am an AI assistant. What's your name?"
        elif "remember" in last_human_message_content.lower():
            response_content = "Yes, I can remember our conversation history. What would you like to discuss?"
        elif "favorite color" in last_human_message_content.lower():
            response_content = "As an AI, I don't have a favorite color, but I find blue quite calming."
        else:
            response_content = f"You said: '{last_human_message_content}'. How can I help further?"
        
        return AIMessage(content=response_content)

# Initialize our mock LLM
llm = MockChatOpenAI()

# In-memory store for session histories
# This dictionary will map session IDs (strings) to ChatMessageHistory objects.
store: Dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Retrieves or creates a ChatMessageHistory for a given session ID."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("Setup complete. Mock LLM and get_session_history function are ready.")


## Your Implementation

Now it's your turn! Using the `llm` (MockChatOpenAI) and `get_session_history` function provided in the setup cell, implement the multi-turn conversational chain as described in the requirements.

Your solution should:
1.  Define a `ChatPromptTemplate` with `MessagesPlaceholder` for `chat_history`.
2.  Create an LCEL chain by piping the prompt to the LLM.
3.  Wrap this chain with `RunnableWithMessageHistory`.
4.  Demonstrate its usage with multiple turns for a single session ID.

```python
# YOUR CODE HERE

# 1. Define the ChatPromptTemplate
# prompt = ...

# 2. Create the LCEL chain
# chain = ...

# 3. Wrap the chain with RunnableWithMessageHistory
# with_message_history = ...

# 4. Test with multiple turns for a single session
# config = {"configurable": {"session_id": "your_session_id"}}
# response1 = with_message_history.invoke({"input": "..."}, config=config)
# print(response1.content)
# response2 = with_message_history.invoke({"input": "..."}, config=config)
# print(response2.content)
# ...
```


In [ ]:
# --- Reference Solution ---

# 1. Define the ChatPromptTemplate
# We use MessagesPlaceholder to dynamically insert chat history into the prompt.
# 'chat_history' is the key that RunnableWithMessageHistory will use to pass the history.
# 'input' is the key for the current user message.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a friendly AI assistant. Keep your responses concise and helpful."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

# 2. Create the LCEL chain
# This simple chain just pipes the processed prompt to the LLM.
chain = prompt | llm

# 3. Wrap the chain with RunnableWithMessageHistory
# This wrapper handles fetching and updating the chat history for each session.
# - `get_session_history`: Our function to retrieve/create history for a session ID.
# - `input_messages_key`: The key in the input dictionary that contains the current user message.
# - `history_messages_key`: The key in the prompt template where the chat history should be inserted.
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("Conversational chain created successfully!")

# 4. Test with multiple turns for a single session
print("\n--- Testing Session 1 ---")
session_id_1 = "user_123"
config_1 = {"configurable": {"session_id": session_id_1}}

# Turn 1
user_input_1 = "Hello, who are you?"
response_1 = with_message_history.invoke({"input": user_input_1}, config=config_1)
print(f"User 1: {user_input_1}")
print(f"AI 1: {response_1.content}")

# Turn 2: Ask a follow-up question, expecting memory
user_input_2 = "Can you remember what I just asked you?"
response_2 = with_message_history.invoke({"input": user_input_2}, config=config_1)
print(f"User 1: {user_input_2}")
print(f"AI 1: {response_2.content}")

# Turn 3: Another follow-up, demonstrating context
user_input_3 = "What is your favorite color?"
response_3 = with_message_history.invoke({"input": user_input_3}, config=config_1)
print(f"User 1: {user_input_3}")
print(f"AI 1: {response_3.content}")

# Verify history for session_id_1
print(f"\n--- History for Session '{session_id_1}' ---")
for msg in store[session_id_1].messages:
    print(f"{type(msg).__name__}: {msg.content}")


print("\n--- Testing Session 2 (should be independent) ---")
session_id_2 = "user_456"
config_2 = {"configurable": {"session_id": session_id_2}}

# Turn 1 for Session 2
user_input_s2_1 = "Hi there!"
response_s2_1 = with_message_history.invoke({"input": user_input_s2_1}, config=config_2)
print(f"User 2: {user_input_s2_1}")
print(f"AI 2: {response_s2_1.content}")

# Turn 2 for Session 2
user_input_s2_2 = "Do you know my name?"
response_s2_2 = with_message_history.invoke({"input": user_input_s2_2}, config=config_2)
print(f"User 2: {user_input_s2_2}")
print(f"AI 2: {response_s2_2.content}")

# Verify history for session_id_2
print(f"\n--- History for Session '{session_id_2}' ---")
for msg in store[session_id_2].messages:
    print(f"{type(msg).__name__}: {msg.content}")

print("\nExercise complete! The multi-turn conversational chain is working as expected, maintaining separate histories for different sessions.")
